# Database walkthrough (`emi.db`)

This notebook shows what the module `src/emi/db.py` does, step by step, using real data.
It creates a small test database, loads **one day** of SMARD data into it (16 November 2026)
and displays the contents of the tables after each step.

**Background.** The SMARD client (`emi.smard`) downloads data as weekly JSON files. For
analysis, a database is more convenient: it can answer targeted questions such as
"all prices in March" without stitching files together each time. The project uses
**SQLite**, a database that lives in a single file and is built into Python (module
`sqlite3`), so no database server is required.

The test database is written to `data/walkthrough.sqlite` and recreated on every run.
The `data/` folder is excluded from git.

**Structure of this notebook**

| part | content |
|---|---|
| Part 1: How the data is stored | sections 1–7: schema, tables, the path of a value from SMARD file to table row, reading data back |
| Part 2: Checking the behaviour | checks A–F: repeated imports, corrections, missing values, time ranges, invalid input |
| Overview | which automated test in `tests/test_db.py` covers which behaviour, and a test run |

In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

from emi import db
from emi.smard import SERIES, SmardClient

DB_PATH = Path("../data/walkthrough.sqlite")
DB_PATH.unlink(missing_ok=True)  # remove the file from a previous run -> fresh start

conn = db.connect(DB_PATH)  # create the file and the (empty) tables
db.sync_series(conn)        # copy the series catalogue from series.py into the database

`db.connect` opens the database file, creating it if necessary, and creates any missing
tables. Existing tables and their data are never touched (`CREATE TABLE IF NOT EXISTS`).
The returned `conn` (*connection*) is the handle through which every read and write goes.

---

# Part 1: How the data is stored

Part 1 only looks and explains: it shows the structure of the database and how real SMARD
data ends up in it. Nothing here is deliberately tested or broken; that follows in Part 2.

## 1. The database schema

A *schema* is the blueprint of a database: which tables exist, which columns they have,
what type of data each column holds, and how the tables relate to each other. In this
project the schema is defined once, as SQL text in the constant `SCHEMA` at the top of
`src/emi/db.py`, and `db.connect` applies it.

### Overview

```
┌──────────────────────────────┐            ┌──────────────────────────────────────┐
│ series                       │            │ observations                         │
├──────────────────────────────┤            ├──────────────────────────────────────┤
│ PK  key        TEXT          │◄───────────┤ FK  series_key  TEXT                 │
│     filter_id  INTEGER       │  1      n  │ PK  resolution  TEXT                 │
│     region     TEXT          │            │ PK  ts_utc      INTEGER              │
│     label      TEXT          │            │     value       REAL                 │
│     unit       TEXT          │            │                                      │
│     kind       TEXT          │            │ PK = (series_key, resolution, ts_utc)│
└──────────────────────────────┘            └──────────────────────────────────────┘
      one row per series                      one row per value
      (~20 rows)                              (hundreds of thousands of rows)
```

`PK` marks the **primary key**, `FK` the **foreign key**. The arrow reads: *each row in
`observations` belongs to exactly one row in `series`; one series has many observations*
(a "one-to-many" relationship, `1 : n`).

### Building blocks

**Data types.** SQLite knows only a few basic types. This schema uses three:

| type | holds | used for |
|---|---|---|
| `TEXT` | character strings | names, units, `'hour'` / `'quarterhour'` |
| `INTEGER` | whole numbers | SMARD filter IDs, timestamps in milliseconds |
| `REAL` | floating-point numbers | the measured values |

**`NOT NULL`.** `NULL` is SQL's marker for "no value". A column declared `NOT NULL` refuses
it, so every stored row is complete. In particular `observations.value` can never be empty:
values that SMARD has not published yet are simply not stored.

**Primary key.** The primary key identifies a row uniquely; no two rows may share it.

* In `series` it is the single column `key`: there can be only one series called `'price'`.
* In `observations` it is the **combination** of three columns (a *composite key*):
  `(series_key, resolution, ts_utc)`. The same point in time may therefore appear several
  times (for different series, or as hour *and* quarter-hour), but each combination only
  once. This rule is what makes the upsert work (check A in Part 2).
* The primary key also acts as an **index**, comparable to the index of a book: looking up
  "price, hour, between March and April" does not require scanning the whole table.

**Foreign key.** `observations.series_key` is declared with `REFERENCES series(key)`. The
database then only accepts a `series_key` that exists in `series`. A typo such as `'prise'`
is rejected with an error instead of silently creating a series nobody reads. (SQLite
enforces this only when `PRAGMA foreign_keys = ON` is set, which `db.connect` does.)

**Why two tables instead of one?** The descriptive information (label, unit, filter ID)
is stored once per series rather than repeated in every one of the many value rows. This
saves space and avoids contradictions; the technical term is *normalisation*.

### The schema as stored in the database

SQLite keeps the original `CREATE TABLE` statements in an internal table called
`sqlite_master`. Reading it back shows exactly what `db.connect` created:

In [2]:
for (sql,) in conn.execute("SELECT sql FROM sqlite_master WHERE type = 'table'"):
    print(sql)
    print()

CREATE TABLE series (
    key        TEXT PRIMARY KEY,      -- project name, e.g. 'price'
    filter_id  INTEGER NOT NULL,      -- SMARD filter ID, e.g. 4169
    region     TEXT NOT NULL,         -- 'DE-LU'
    label      TEXT NOT NULL,
    unit       TEXT NOT NULL,         -- 'EUR/MWh' or 'MWh'
    kind       TEXT NOT NULL          -- 'price', 'actual' or 'forecast'
)

CREATE TABLE observations (
    series_key TEXT NOT NULL REFERENCES series(key),
    resolution TEXT NOT NULL,         -- 'hour' or 'quarterhour'
    ts_utc     INTEGER NOT NULL,      -- interval start, UTC epoch milliseconds
    value      REAL NOT NULL,
    -- One value per series, resolution and point in time. This key is what
    -- makes the upsert below work.
    PRIMARY KEY (series_key, resolution, ts_utc)
)



The command `PRAGMA table_info` lists the columns of a table in tabular form. The column
`notnull` is 1 for `NOT NULL` columns; `pk` gives the position of a column within the
primary key (0 = not part of it).

In [3]:
pd.read_sql("PRAGMA table_info(observations)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,series_key,TEXT,1,None,1
1,1,resolution,TEXT,1,None,2
2,2,ts_utc,INTEGER,1,None,3
3,3,value,REAL,1,None,0


`PRAGMA foreign_key_list` shows the foreign key: column `series_key` (`from`) points to
column `key` (`to`) of table `series`.

In [4]:
pd.read_sql("PRAGMA foreign_key_list(observations)", conn)[["table", "from", "to"]]

,table,from,to
0,series,series_key,key


## 2. Table `series`: the catalogue

The database has two tables. The first one, `series`, lists every time series used in the
project, one row per series. Its content is a copy of the catalogue in
`src/emi/smard/series.py`:

| column | meaning |
|---|---|
| `key` | short project name, used everywhere in the code |
| `filter_id` | SMARD's numeric ID of the series |
| `unit` | `EUR/MWh` for prices, `MWh` for energy quantities |
| `kind` | `price` (auction result), `actual` (measured afterwards) or `forecast` (known before the auction) |

In [5]:
pd.read_sql("SELECT * FROM series", conn)

,key,filter_id,region,label,unit,kind
0,price,4169,DE-LU,Day-ahead price DE-LU,EUR/MWh,price
1,load,410,DE-LU,Grid load,MWh,actual
2,residual_load,4359,DE-LU,Residual load,MWh,actual
3,wind_onshore,4067,DE-LU,Wind onshore,MWh,actual
4,wind_offshore,1225,DE-LU,Wind offshore,MWh,actual
5,solar,4068,DE-LU,Photovoltaics,MWh,actual
6,biomass,4066,DE-LU,Biomass,MWh,actual
7,hydro,1226,DE-LU,Hydropower,MWh,actual
8,other_renewable,1228,DE-LU,Other renewables,MWh,actual
9,lignite,1223,DE-LU,Lignite,MWh,actual


## 3. Loading real data

The next cell downloads one day (16 November 2026, German time) and stores it:

* **price** in hourly resolution: 24 values
* **load** (grid load, i.e. electricity demand) in hourly *and* quarter-hourly resolution:
  24 + 96 values

`upsert_observations` expects a list of pairs `(timestamp in ms, value)`, the format in
which SMARD delivers its data. `fetch_range` returns a pandas series instead, so the small
helper `to_pairs` converts between the two.

Downloaded files are cached in `data/smard_cache`, so a repeated run needs hardly any
network access.

In [6]:
client = SmardClient(cache_dir=Path("../data/smard_cache"))
start = pd.Timestamp("2026-11-16", tz="Europe/Berlin")
end = start + pd.Timedelta(days=1)


def to_pairs(values: pd.Series) -> list[tuple[int, float]]:
    """pandas series -> [(timestamp_ms, value), ...]"""
    return [(int(ts.value // 1_000_000), v) for ts, v in values.items()]


for key, resolution in [("price", "hour"), ("load", "hour"), ("load", "quarterhour")]:
    values = client.fetch_range(SERIES[key].filter_id, start, end, resolution=resolution)
    n = db.upsert_observations(conn, key, resolution, to_pairs(values))
    print(f"{key:<6} {resolution:<12} {n:>3} values stored")

price  hour          24 values stored


load   hour          24 values stored


load   quarterhour   96 values stored


## 4. Table `observations`: the raw values

The second table holds the actual numbers, one row per value:

| column | meaning |
|---|---|
| `series_key` | which series the value belongs to (refers to `series.key`) |
| `resolution` | `hour` or `quarterhour` |
| `ts_utc` | start of the interval, in milliseconds since 1970-01-01 00:00 UTC |
| `value` | the value itself |

Storing time as a plain number in UTC keeps the database free of time zones, so daylight
saving time cannot cause errors at this level. The drawback is that the numbers are hard to
read. For display only, the helper `with_time` adds a column with German local time; this
column does not exist in the database.

In [7]:
def with_time(df: pd.DataFrame) -> pd.DataFrame:
    """Add a readable German-time column next to ts_utc (display only)."""
    utc = pd.to_datetime(df["ts_utc"], unit="ms", utc=True)
    df["time_berlin"] = utc.dt.tz_convert("Europe/Berlin")
    return df


with_time(pd.read_sql("SELECT * FROM observations WHERE series_key = 'price' LIMIT 6", conn))

,series_key,resolution,ts_utc,value,time_berlin
0,price,hour,1794783600000,135.65,2026-11-16 00:00:00+01:00
1,price,hour,1794787200000,124.99,2026-11-16 01:00:00+01:00
2,price,hour,1794790800000,119.78,2026-11-16 02:00:00+01:00
3,price,hour,1794794400000,114.10,2026-11-16 03:00:00+01:00
4,price,hour,1794798000000,115.56,2026-11-16 04:00:00+01:00
5,price,hour,1794801600000,129.69,2026-11-16 05:00:00+01:00


The first row is midnight in Germany (`00:00+01:00`), which is 23:00 UTC on the previous
day. That is why `ts_utc` does not correspond to "00:00" when converted naively.

Number of rows per series and resolution:

In [8]:
pd.read_sql(
    """SELECT series_key, resolution, COUNT(*) AS n_rows
       FROM observations
       GROUP BY series_key, resolution""",
    conn,
)

,series_key,resolution,n_rows
0,load,hour,24
1,load,quarterhour,96
2,price,hour,24


## 5. From SMARD file to table row

This section follows **one single value** from the original SMARD file into the database:
the day-ahead price for 16 November 2026, 00:00–01:00 German time.

### The path at a glance

```
SMARD file  4169_DE-LU_hour_1794783600000.json
            │       │     │         │
            │       │     │         └─ start of the week covered by the file
            │       │     └─────────── resolution          ──► column resolution
            │       └───────────────── region (always DE-LU)
            └───────────────────────── filter ID 4169      ──► series.py: 4169 = 'price'
                                                               ──► column series_key
  content:  {"series": [[1794783600000, 135.65], [1794787200000, 124.99], ...]}
                         └──────┬──────┘  └─┬──┘
                          time stamp      value
                                │           │
                                ▼           ▼
                        column ts_utc   column value

row in observations:  ('price', 'hour', 1794783600000, 135.65)
```

The key observation: a SMARD file contains **only two of the four columns** (time stamp
and value). The other two, *which series* and *which resolution*, are not inside the data
at all; they follow from which file was requested. That is why `upsert_observations` takes
`series_key` and `resolution` as separate arguments and the `(timestamp, value)` pairs as
a list.

### Step 1: the raw file

SMARD splits every series into weekly files. The week of 16 November starts on Monday
16 November, 00:00 German time. The cell below finds that file in the index of the price
series, reads it from the local cache and prints its beginning.

In [9]:
filter_id = SERIES["price"].filter_id  # 4169
start_ms = int(start.value // 1_000_000)  # 16 Nov 00:00 German time, as UTC milliseconds

# All weekly files of the hourly price series; pick the one that contains start_ms
week_starts = client.chunk_timestamps(filter_id, "DE-LU", "hour")
week = max(ts for ts in week_starts if ts <= start_ms)
pairs = client.fetch_chunk(filter_id, "DE-LU", "hour", week)  # also stores the file in the cache

file_name = f"{filter_id}_DE-LU_hour_{week}.json"
raw = json.loads((client.cache_dir / file_name).read_text())

print("file name:          ", file_name)
print("fields in the file: ", list(raw))
print("entries in 'series':", len(raw["series"]), "(7 days x 24 hours)")
print("first 3 entries:    ", raw["series"][:3])

file name:           4169_DE-LU_hour_1794783600000.json
fields in the file:  ['meta_data', 'series']
entries in 'series': 168 (7 days x 24 hours)
first 3 entries:     [[1794783600000, 135.65], [1794787200000, 124.99], [1794790800000, 119.78]]


Each entry of `series` is a list `[time stamp, value]`. The file has 168 entries,
one per hour of the week. (`meta_data` is information about the file itself and is not
used.)

### Step 2: the pairs

`SmardClient.fetch_chunk` reads the same file and returns the entries as Python tuples:
time stamps as whole numbers, values as floating-point numbers, `null` as `None`. This is
exactly the input format of `upsert_observations`.

In [10]:
pairs[:3]

[(1794783600000, 135.65), (1794787200000, 124.99), (1794790800000, 119.78)]

### Step 3: the row

`upsert_observations(conn, "price", "hour", pairs)` adds the two missing pieces of
information to every pair and writes one row per pair. The table below shows where each
column of the first row comes from, next to the row as it is actually stored:

In [11]:
ts, value = pairs[0]
pd.DataFrame(
    {
        "column": ["series_key", "resolution", "ts_utc", "value"],
        "stored value": ["price", "hour", ts, value],
        "comes from": [
            "filter ID 4169 in the file name, translated via series.py",
            "'hour' in the file name",
            "1st number of the entry",
            "2nd number of the entry",
        ],
    }
)

,column,stored value,comes from
0,series_key,price,"filter ID 4169 in the file name, translated vi..."
1,resolution,hour,'hour' in the file name
2,ts_utc,1794783600000,1st number of the entry
3,value,135.65,2nd number of the entry


In [12]:
pd.read_sql(
    "SELECT * FROM observations WHERE series_key = 'price' AND resolution = 'hour' AND ts_utc = ?",
    conn,
    params=[ts],
)

,series_key,resolution,ts_utc,value
0,price,hour,1794783600000,135.65


### One point in time, three rows

Grid load was loaded in addition to the price, in two resolutions. The same moment
(16 Nov, 00:00) therefore appears in **three** rows. They differ in `series_key` or
`resolution`, so the composite primary key `(series_key, resolution, ts_utc)` is different
for each of them and all three may exist side by side:

In [13]:
pd.read_sql("SELECT * FROM observations WHERE ts_utc = ?", conn, params=[ts])

,series_key,resolution,ts_utc,value
0,price,hour,1794783600000,135.65
1,load,hour,1794783600000,45648.65
2,load,quarterhour,1794783600000,11493.70


### Connecting the two tables (JOIN)

A row in `observations` carries only the short name `'price'` or `'load'`. Label and
unit are stored once, in `series`. A **JOIN** combines both tables at query time: for
every row in `observations`, SQLite looks up the row in `series` whose `key` equals
`series_key` and puts the columns next to each other. This is the foreign key relationship
from section 1 in action.

In [14]:
pd.read_sql(
    """SELECT o.series_key, s.label, s.unit, o.resolution, o.ts_utc, o.value
       FROM observations AS o
       JOIN series AS s ON s.key = o.series_key
       WHERE o.ts_utc = ?""",
    conn,
    params=[ts],
)

,series_key,label,unit,resolution,ts_utc,value
0,price,Day-ahead price DE-LU,EUR/MWh,hour,1794783600000,135.65
1,load,Grid load,MWh,hour,1794783600000,45648.65
2,load,Grid load,MWh,quarterhour,1794783600000,11493.70


### Long format vs. wide format

`observations` is a **long** table: one value per row, with the series name as a column.
New series can be added without changing the table structure, which suits raw data.

For analysis, a **wide** table is more convenient: one row per hour and one column per
series. The cell below converts the hourly values with `pivot`. A table of this shape,
cleaned and complete, is what the later module `cleaning.py` will produce (`hourly_clean`).

In [15]:
long = pd.read_sql(
    "SELECT series_key, ts_utc, value FROM observations WHERE resolution = 'hour'", conn
)
wide = long.pivot(index="ts_utc", columns="series_key", values="value")
wide.index = pd.to_datetime(wide.index, unit="ms", utc=True).tz_convert("Europe/Berlin")
wide.head(4)

series_key,load,price
ts_utc,,
2026-11-16 00:00:00+01:00,45648.65,135.65
2026-11-16 01:00:00+01:00,44919.31,124.99
2026-11-16 02:00:00+01:00,44620.41,119.78
2026-11-16 03:00:00+01:00,44287.33,114.10


## 6. Hourly and quarter-hourly values side by side

Hourly and quarter-hourly values of the same series are stored next to each other without
interfering, because the resolution is part of each row's identity.

The cell below shows grid load between 00:00 and 01:00: once as four quarter-hour values
and once as one hourly value. For energy quantities, the hourly value is the **sum** of the
four quarter-hours (for prices it would be the **mean**). The tiny difference is rounding.
This conversion is the job of the upcoming module `cleaning.py`.

In [16]:
first_hour = (start, start + pd.Timedelta(hours=1))
qh = db.read_observations(conn, "load", "quarterhour", *first_hour)
h = db.read_observations(conn, "load", "hour", *first_hour)

# Convert the index to German time for display (the stored values are UTC)
display(qh.tz_convert("Europe/Berlin").to_frame("quarterhour_mwh"))
print(f"Sum of the 4 quarter-hours: {qh.sum():,.2f} MWh")
print(f"Hourly value from SMARD:    {h.iloc[0]:,.2f} MWh")

,quarterhour_mwh
2026-11-16 00:00:00+01:00,11493.70
2026-11-16 00:15:00+01:00,11412.54
2026-11-16 00:30:00+01:00,11381.42
2026-11-16 00:45:00+01:00,11361.00


Sum of the 4 quarter-hours: 45,648.66 MWh
Hourly value from SMARD:    45,648.65 MWh


## 7. Reading data back with `read_observations`

The rest of the project does not write SQL itself; it calls `read_observations`, which
returns a pandas series with a UTC time index.

The time range is **half-open**, written `[start, end)`: `start` is included, `end` is not.
A request for 06:00 to 10:00 therefore returns the hours 06, 07, 08 and 09. Consecutive
ranges can be chained this way without gaps or overlaps.

In [17]:
prices = db.read_observations(
    conn, "price", "hour",
    start + pd.Timedelta(hours=6), start + pd.Timedelta(hours=10),
)
prices.tz_convert("Europe/Berlin").to_frame("price_eur_mwh")  # German time for display

,price_eur_mwh
2026-11-16 06:00:00+01:00,174.35
2026-11-16 07:00:00+01:00,214.16
2026-11-16 08:00:00+01:00,225.86
2026-11-16 09:00:00+01:00,220.53


---

# Part 2: Checking the behaviour

Part 1 described *how* the data is stored. Part 2 checks that `emi.db` actually
**behaves** as promised, including in situations that go wrong on purpose.

Every check follows the same pattern:

* **Expected**: the promised behaviour, in one sentence
* **Automated test**: the test in `tests/test_db.py` that checks the same thing
* a code cell that tries it out and ends with `✓` if the behaviour is as expected
  (an `assert` statement stops the cell with an error otherwise)

**Notebook checks vs. automated tests.** The checks here use the real data from Part 1
and are run by hand, to *see* what happens. The automated tests check the same
properties with a few made-up values in a throwaway in-memory database. They run in under
a second, need no network, and run automatically on every change (locally with `pytest`,
on GitHub via CI). The overview at the end connects both.

In [18]:
def count_rows() -> int:
    """Number of rows in observations."""
    return conn.execute("SELECT COUNT(*) FROM observations").fetchone()[0]


t0 = int(start.value // 1_000_000)  # 16 Nov 00:00 German time, as UTC milliseconds

### Check A: importing twice creates no duplicates

**Expected:** storing the same values a second time leaves the table unchanged.
**Automated test:** `test_upsert_twice_changes_nothing`

"Upsert" combines **up**date and in**sert**: a row is inserted if it does not exist yet
and updated if it does. Whether it "already exists" is decided by the primary key
`(series_key, resolution, ts_utc)`. An operation that can be repeated without changing
the result is called *idempotent*. This matters because imports will run regularly and
may overlap with data that is already stored.

In [19]:
prices = client.fetch_range(SERIES["price"].filter_id, start, end)
before = count_rows()
db.upsert_observations(conn, "price", "hour", to_pairs(prices))
after = count_rows()

print(f"rows before: {before}, rows after: {after}")
assert before == after
print("✓ no duplicates")

rows before: 144, rows after: 144
✓ no duplicates


### Check B: a corrected value replaces the old one

**Expected:** if a value arrives again with a different number, the new number is kept.
**Automated test:** `test_upsert_replaces_corrected_value`

SMARD occasionally revises published values. The correction is simulated here by writing
an artificial price of 999 for midnight.

In [20]:
db.upsert_observations(conn, "price", "hour", [(t0, 999.0)])

display(with_time(pd.read_sql(
    "SELECT * FROM observations WHERE series_key = 'price' LIMIT 3", conn
)))
assert db.read_observations(conn, "price", "hour").iloc[0] == 999.0
print("✓ new value replaced the old one")

,series_key,resolution,ts_utc,value,time_berlin
0,price,hour,1794783600000,999.00,2026-11-16 00:00:00+01:00
1,price,hour,1794787200000,124.99,2026-11-16 01:00:00+01:00
2,price,hour,1794790800000,119.78,2026-11-16 02:00:00+01:00


✓ new value replaced the old one


### Check C: missing values are not stored

**Expected:** a value of `None` (SMARD's `null`, "not published yet") is skipped.
**Automated test:** `test_upsert_skips_none`

SMARD files for the current week contain `null` for hours that lie in the future. Storing
them would fill the table with empty rows; instead, "no row" means "no value".

In [21]:
next_day = t0 + 24 * 3_600_000  # 17 Nov 00:00, not loaded so far
before = count_rows()
written = db.upsert_observations(conn, "price", "hour", [(next_day, None)])

print(f"values written: {written}, rows before: {before}, rows after: {count_rows()}")
assert written == 0
assert count_rows() == before
print("✓ None was skipped")

values written: 0, rows before: 144, rows after: 144
✓ None was skipped


### Check D: the time range excludes its end

**Expected:** `read_observations(start, end)` returns values from `start` up to, but not
including, `end`.
**Automated test:** `test_read_range_is_half_open_and_utc`

Section 7 asked for 06:00 to 10:00; the result must be exactly four hours, the last one
starting at 09:00.

In [22]:
result = db.read_observations(
    conn, "price", "hour", start + pd.Timedelta(hours=6), start + pd.Timedelta(hours=10)
)
last = result.index[-1].tz_convert("Europe/Berlin")

print("number of hours:    ", len(result))
print("last hour starts at:", last)
assert len(result) == 4
assert last.hour == 9
print("✓ end is excluded")

number of hours:     4
last hour starts at: 2026-11-16 09:00:00+01:00
✓ end is excluded


### Check E: a time without time zone is rejected

**Expected:** a timestamp without time zone raises a `ValueError` with a clear message.
**Automated test:** `test_read_rejects_naive_timestamp`

`2026-11-16 06:00` alone could mean German time or UTC, which differ by one or two hours.
Guessing would silently shift all results, so the function refuses. The `try`/`except`
block catches the error in order to display it.

In [23]:
try:
    db.read_observations(conn, "price", "hour", start=pd.Timestamp("2026-11-16 06:00"))
except ValueError as err:
    print("ValueError:", err)
    print("✓ rejected")
else:
    raise AssertionError("expected a ValueError")

ValueError: timestamp 2026-11-16 06:00:00 must be timezone-aware
✓ rejected


### Check F: an unknown series is rejected

**Expected:** values for a series that is not in the `series` table are refused by the
database.
**Automated test:** `test_unknown_series_is_rejected`

This is the foreign key from section 1 at work. A typo such as `'prise'` raises an
`IntegrityError` instead of creating a series that nothing else knows about.

In [24]:
try:
    db.upsert_observations(conn, "prise", "hour", [(t0, 1.0)])
except sqlite3.IntegrityError as err:
    print("IntegrityError:", err)
    print("✓ rejected")
else:
    raise AssertionError("expected an IntegrityError")

IntegrityError: FOREIGN KEY constraint failed
✓ rejected


---

# Overview: notebook and automated tests

The table lists every test in `tests/test_db.py`, what it checks, and where the same
behaviour appears in this notebook.

| automated test | checks that ... | in this notebook |
|---|---|---|
| `test_connect_creates_file_and_is_reopenable` | the file (and its folder) is created and data survives closing and reopening | setup cell at the top |
| `test_sync_series_is_idempotent` | loading the catalogue twice works and `price` has ID 4169 | section 2 |
| `test_hour_and_quarterhour_live_side_by_side` | both resolutions of one series are stored separately | sections 5 and 6 |
| `test_upsert_twice_changes_nothing` | no duplicates on repeated import | check A |
| `test_upsert_replaces_corrected_value` | corrections overwrite old values | check B |
| `test_upsert_skips_none` | `None` is not stored | check C |
| `test_read_range_is_half_open_and_utc` | the end of a range is excluded; times in other zones are converted | section 7, check D |
| `test_read_rejects_naive_timestamp` | times without time zone are refused | check E |
| `test_unknown_series_is_rejected` | the foreign key blocks unknown series | check F |

The cell below runs the automated tests, the same way as `pytest tests/test_db.py -v`
in a terminal. `-v` (*verbose*) prints one line per test.

In [25]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "../tests/test_db.py", "-v", "--no-header", "--color=no"],
    capture_output=True,
    text=True,
)
print(result.stdout)

============================= test session starts ==============================
collecting ... collected 9 items

../tests/test_db.py::test_connect_creates_file_and_is_reopenable PASSED  [ 11%]
../tests/test_db.py::test_sync_series_is_idempotent PASSED               [ 22%]
../tests/test_db.py::test_upsert_twice_changes_nothing PASSED            [ 33%]
../tests/test_db.py::test_upsert_replaces_corrected_value PASSED         [ 44%]
../tests/test_db.py::test_upsert_skips_none PASSED                       [ 55%]
../tests/test_db.py::test_hour_and_quarterhour_live_side_by_side PASSED  [ 66%]
../tests/test_db.py::test_read_range_is_half_open_and_utc PASSED         [ 77%]
../tests/test_db.py::test_read_rejects_naive_timestamp PASSED            [ 88%]
../tests/test_db.py::test_unknown_series_is_rejected PASSED              [100%]

============================== 9 passed in 0.60s ===============================



## Summary

* `connect` creates or opens the database, `sync_series` fills the catalogue.
* `observations` stores raw values in their native resolution, with time as UTC milliseconds.
  Series and resolution come from the requested file, time stamp and value from its content.
* `upsert_observations` is safe to repeat, picks up corrections and skips missing values.
* `read_observations` returns a pandas series for a half-open time range and refuses
  ambiguous times.

In [26]:
conn.close()